# Read-only CPU root-cause diagnostics: training/validation ONLY

Use **accelerator None**. Do not run notebooks 13/14 for this task: those helpers read mixed provenance/catalogs. This notebook reads only training prefixes, prepared training arrays, train tokens and existing checkpoints. It does not modify Drive, train, extract, download model weights, read final-test identities/results, or evaluate the final test.

1. Read `docs/LOW_VALIDATION_ROOT_CAUSE_AUDIT.md` first.
2. Set `RUN_TRAIN_VALIDATION_DIAGNOSTICS=True` and run the single cell. Leave `RUN_CPU_VALIDATION_INFERENCE=False` initially: it reports row-map structure, pool counts, feature statistics, token/pooled agreement, temperature and existing histories.
3. Only if needed, enable `RUN_CPU_VALIDATION_INFERENCE=True` to re-score the validation rows with frozen best states on CPU. Patch inference may be slow. Same-concept predictions and paired bootstrap intervals are diagnostic additions, not replacements for exact-image accuracy; CPU FP32 can differ from recorded GPU autocast scores.

No output files are saved. Copy only the summary you wish to share. The module fails closed on unexpected JSON order instead of loading final-test fields. It cannot independently prove original EEG trial identities without training-only upstream evidence. The 0.5% and 5% chance baselines apply respectively to exact-image and same-concept-top-image diagnostics when the observed pool is 20 concepts x10 images.


In [ ]:
RUN_TRAIN_VALIDATION_DIAGNOSTICS = False #@param {type:"boolean"}
assert RUN_TRAIN_VALIDATION_DIAGNOSTICS, 'Read the audit scope, then explicitly enable CPU diagnostics.'
RUN_CPU_VALIDATION_INFERENCE = False #@param {type:"boolean"}
import os, sys, json, subprocess, tempfile
from pathlib import Path
import torch
assert not torch.cuda.is_available(), 'Select hardware accelerator None. This audit must not run on a GPU runtime.'
from google.colab import drive
drive.mount('/content/drive')
CODE_REVISION = 'ee22fa0b9ee83924a95fea82a27708f20b902bb9'
REPO=Path(tempfile.mkdtemp(prefix='validation-root-audit-',dir='/content'))/'repo'
subprocess.run(['git','clone','--no-checkout','https://github.com/firozehrazavi/eeg-alignment-phase1.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',CODE_REVISION],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==CODE_REVISION
WORK=Path('/content/drive/MyDrive/eeg_results/patch_phase1')
args=[sys.executable,'-m','phase1.validation_root_audit',
 '--prepared',str(WORK/'pilot_sub01_200concepts_v1'),
 '--training',str(WORK/'controlled_sub01_five_epochs_v1')]
if RUN_CPU_VALIDATION_INFERENCE:args+=['--validation-inference']
print('Read-only CPU diagnostics; no optimizer, test data, output directory or Drive writes.')
print('Optional CPU Patch validation can be slow. Statistics mode is the default.')
result=subprocess.run(args,cwd=REPO,env=dict(os.environ,CUDA_VISIBLE_DEVICES='',OMP_NUM_THREADS='4',MKL_NUM_THREADS='4'),
 capture_output=True,text=True,timeout=3600 if RUN_CPU_VALIDATION_INFERENCE else 1200)
if result.stderr:print(result.stderr)
if result.returncode:raise RuntimeError('Audit failed; inputs preserved. Review the traceback above, do not weaken checks.')
report=json.loads(result.stdout)
print(json.dumps(report,indent=2))
print('STOP. Validation diagnostics only; final-test evaluation remains NO-GO pending review.')
